# ML-08 — Capstone Modeling Lane

**Lane:** Refresh / Content Opportunity Scoring. This notebook follows the ML-04 warehouse contract and compares a learned ranking with the ML-07 rule on the same client-held-out candidate set.

## 1. Method choice and why

I use **L2-regularized Logistic Regression** to estimate the probability that a candidate meets the next-month impression-decline proxy. The probabilities provide a ranking for editorial review, while the standardized coefficients are inspectable. A more complex tree ensemble is not justified unless it improves the held-out ranking enough to outweigh the added complexity.

The evaluation follows the contract already used in ML-04 and ML-07: March 2026 signals predict the retrospective April proxy. A candidate is a March page-client item that satisfies the ML-07 rule (at least 500 impressions, average position 1–20, CTR below 0.5%). The comparison asks whether the model can **reorder that same review pool** more usefully than the rule's `log1p(March impressions)` ranking; it does not assess items the rule excludes.

I also reviewed the published capstone paper linked from `submission/paper_url.txt`. Its client-held-out design and warning that overlapping label/features do not establish early warning are relevant. Its starter-snapshot target, rule, and reported metrics differ, so they are not reused here.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

repo_root = Path.cwd()
while not (repo_root / '.env').is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / '.env').is_file():
    raise FileNotFoundError('Repository-root .env with HF_TOKEN is required; the token is never displayed.')
hf_token = None
for line in (repo_root / '.env').read_text(encoding='utf-8').splitlines():
    if '=' in line and not line.lstrip().startswith('#'):
        key, value = line.split('=', 1)
        if key.strip() == 'HF_TOKEN':
            hf_token = value.strip().strip(chr(34)).strip(chr(39))
            break
if not hf_token or not hf_token.startswith('hf_'):
    raise ValueError('HF_TOKEN is missing or malformed in .env; token value was not displayed.')
con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute('CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)', [hf_token])
del hf_token
warehouse = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
march_source = 'read_parquet(' + repr(warehouse + '/month=2026-03/*.parquet') + ')'
april_source = 'read_parquet(' + repr(warehouse + '/month=2026-04/*.parquet') + ')'
print('Warehouse access configured from the ignored .env; the credential was not displayed.')

Warehouse access configured from the ignored .env; the credential was not displayed.


In [2]:
feature_sql = f'''
WITH march AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_mar,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_mar,
           SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS clicks_mar,
           SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS position_sum_mar
    FROM {march_source}
    GROUP BY client_hash_id, content_hash_id
), april AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_days_apr,
           SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions_apr
    FROM {april_source}
    GROUP BY client_hash_id, content_hash_id
)
SELECT march.client_hash_id, march.content_hash_id,
       march.gsc_days_mar, march.impressions_mar, march.clicks_mar,
       march.position_sum_mar / NULLIF(march.impressions_mar, 0) AS avg_position_mar,
       100.0 * march.clicks_mar / NULLIF(march.impressions_mar, 0) AS ctr_mar,
       CAST(april.impressions_apr <= 0.8 * march.impressions_mar AS INTEGER) AS april_decline_label
FROM march INNER JOIN april USING (client_hash_id, content_hash_id)
WHERE march.gsc_days_mar >= 20 AND april.gsc_days_apr >= 20
  AND march.impressions_mar > 0
'''
items = con.execute(feature_sql).df()
con.close()
ID_COLUMNS = ['client_hash_id', 'content_hash_id']
LABEL = 'april_decline_label'
assert items[ID_COLUMNS].duplicated().sum() == 0
assert items[LABEL].isin([0, 1]).all()
print(f'Paired, adequately observed March-April items: {len(items):,}')
print(f'April decline-proxy rate across all eligible items: {items[LABEL].mean():.3f}')
print('Grain: one row per pseudonymous client-content pair; GSC availability was filtered with IS TRUE.')

Paired, adequately observed March-April items: 95,633
April decline-proxy rate across all eligible items: 0.495
Grain: one row per pseudonymous client-content pair; GSC availability was filtered with IS TRUE.


## 2. Split design

Use one reproducible **client-group holdout** (seed 42, 20% of client groups held out). A client and all its content items appear in only one side, preventing related pages from the same client leaking across train and test. The target is the April 20%-decline proxy, observed only for evaluation/training; the feature window is March. The held-out test set is not used for fitting or threshold selection. Because this is one split, its metrics are a directional estimate, not a stable generalization guarantee.

In [3]:
candidate_mask = (
    (items['impressions_mar'] >= 500)
    & (items['avg_position_mar'] > 0)
    & (items['avg_position_mar'] <= 20)
    & (items['ctr_mar'] < 0.5)
)
candidates = items.loc[candidate_mask].copy().reset_index(drop=True)
FEATURES = ['log_impressions_mar', 'log_clicks_mar', 'ctr_mar', 'avg_position_mar', 'gsc_days_mar']
candidates['log_impressions_mar'] = np.log1p(candidates['impressions_mar'])
candidates['log_clicks_mar'] = np.log1p(candidates['clicks_mar'])

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(candidates[FEATURES], candidates[LABEL], groups=candidates['client_hash_id']))
train = candidates.iloc[train_idx].copy().reset_index(drop=True)
test = candidates.iloc[test_idx].copy().reset_index(drop=True)
train_clients = set(train['client_hash_id'])
test_clients = set(test['client_hash_id'])
assert train_clients.isdisjoint(test_clients)
assert len(test) >= 50
assert train[LABEL].nunique() == 2 and test[LABEL].nunique() == 2
print(f'ML-07 rule candidates: {len(candidates):,}')
print(f'Train: {len(train):,} candidates from {len(train_clients)} clients; test: {len(test):,} from {len(test_clients)} clients.')
print(f'Client overlap: {len(train_clients.intersection(test_clients))}; held-out candidate decline-proxy base rate: {test[LABEL].mean():.3f}')
print('Identifiers are used only for grouping and are not model features.')

ML-07 rule candidates: 39,545


Train: 20,461 candidates from 24 clients; test: 19,084 from 7 clients.
Client overlap: 0; held-out candidate decline-proxy base rate: 0.554
Identifiers are used only for grouping and are not model features.


## 3. Train + compare vs my baseline

Fit Logistic Regression on the training clients only. Compare its probability ranking with the exact ML-07 rule ranking on the **same held-out candidates**, label, and evaluation metrics. Precision@K answers the practical review-queue question; ROC-AUC and average precision summarize ordering over the entire candidate test set. The test-set positive rate is the reference for interpreting Precision@K.

In [4]:
model = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000, random_state=42),
)
X_train = train[FEATURES]
y_train = train[LABEL].to_numpy(dtype=int)
X_test = test[FEATURES]
y_test = test[LABEL].to_numpy(dtype=int)
model.fit(X_train, y_train)
test['model_score'] = model.predict_proba(X_test)[:, 1]

# Same ML-07 candidate gate and same rule ordering: log impressions, then clicks, then position.
test['baseline_score'] = np.log1p(test['impressions_mar'])
baseline_ranked = test.sort_values(
    ['baseline_score', 'clicks_mar', 'avg_position_mar'],
    ascending=[False, False, True], kind='mergesort',
).reset_index(drop=True)
model_ranked = test.sort_values(
    ['model_score', 'baseline_score', 'clicks_mar', 'avg_position_mar'],
    ascending=[False, False, False, True], kind='mergesort',
).reset_index(drop=True)

def precision_at_k(ranked, k):
    return float(ranked[LABEL].head(k).mean())

results = []
for name, ranked, scores in [
    ('ML-07 rule: log impressions', baseline_ranked, baseline_ranked['baseline_score']),
    ('Logistic Regression', model_ranked, model_ranked['model_score']),
]:
    results.append({
        'Ranking': name,
        'Test candidates': len(test),
        'Test base rate': y_test.mean(),
        'Precision@10': precision_at_k(ranked, 10),
        'Precision@20': precision_at_k(ranked, 20),
        'Precision@50': precision_at_k(ranked, 50),
        'ROC-AUC': roc_auc_score(y_test, scores),
        'Average precision': average_precision_score(y_test, scores),
    })
comparison = pd.DataFrame(results).set_index('Ranking')
print('Held-out comparison — same client groups, candidate set, label, and metrics:')
display(comparison.round(3))
print('No April values or label-derived fields are used as features or ranking inputs.')

Held-out comparison — same client groups, candidate set, label, and metrics:


,Test candidates,Test base rate,Precision@10,Precision@20,Precision@50,ROC-AUC,Average precision
Ranking,,,,,,,
ML-07 rule: log impressions,19084,0.554,0.4,0.30,0.32,0.496,0.55
Logistic Regression,19084,0.554,0.8,0.75,0.72,0.496,0.55


No April values or label-derived fields are used as features or ranking inputs.


## 4. Errors and interpretation

The coefficients below are for standardized features: a positive coefficient is associated with higher modelled decline odds within this candidate pool; it is not a causal effect. Then inspect up to three false positives and three false negatives at the model's fixed 0.5 classification threshold. Cases are anonymous and show only March aggregate signals.

In [5]:
classifier = model.named_steps['logisticregression']
coefficients = pd.Series(classifier.coef_[0], index=FEATURES)
coefficients = coefficients.reindex(coefficients.abs().sort_values(ascending=False).index)
print('Largest standardized Logistic Regression coefficients:')
display(coefficients.rename('standardized_coefficient').head(5).to_frame().round(3))

# This threshold is descriptive only; ranking metrics above do not depend on it.
test['predicted_at_0_5'] = (test['model_score'] >= 0.5).astype(int)
test['model_rank'] = test['model_score'].rank(method='first', ascending=False).astype(int)
false_positives = test.loc[(test['predicted_at_0_5'] == 1) & (test[LABEL] == 0)].sort_values('model_score', ascending=False).head(3).copy()
false_negatives = test.loc[(test['predicted_at_0_5'] == 0) & (test[LABEL] == 1)].sort_values('model_score', ascending=True).head(3).copy()
error_rows = []
for error_type, rows in [('False positive', false_positives), ('False negative', false_negatives)]:
    for _, row in rows.iterrows():
        why_hard = (
            'March review signals were present, but April impressions did not cross the decline threshold.'
            if error_type == 'False positive'
            else 'The April threshold decline was not separated by these March aggregate features.'
        )
        error_rows.append({
            'Error': error_type,
            'Anonymous rank': int(row['model_rank']),
            'Model probability': row['model_score'],
            'April proxy': int(row[LABEL]),
            'March impressions': int(row['impressions_mar']),
            'March CTR (%)': row['ctr_mar'],
            'March position': row['avg_position_mar'],
            'Observed GSC days': int(row['gsc_days_mar']),
            'Why difficult': why_hard,
        })
error_cases = pd.DataFrame(error_rows)
print(f"Threshold errors on held-out candidates: {(test['predicted_at_0_5'] != test[LABEL]).sum():,} of {len(test):,}; showing up to three of each type.")
if error_cases.empty:
    print('No 0.5-threshold error cases were available to display.')
else:
    display(error_cases.round({'Model probability': 3, 'March CTR (%)': 3, 'March position': 2}))
print('These are errors against an observational month-to-month proxy, not confirmed editorial mistakes.')

Largest standardized Logistic Regression coefficients:


,standardized_coefficient
gsc_days_mar,0.452
ctr_mar,-0.443
avg_position_mar,-0.344
log_clicks_mar,-0.157
log_impressions_mar,-0.062


Threshold errors on held-out candidates: 8,171 of 19,084; showing up to three of each type.

,Error,Anonymous rank,Model probability,April proxy,March impressions,March CTR (%),March position,Observed GSC days,Why difficult
0,False positive,9,0.856,0,676,0.000,0.58,31,"March review signals were present, but April i..."
1,False positive,10,0.856,0,609,0.000,0.67,31,"March review signals were present, but April i..."
2,False positive,12,0.856,0,607,0.000,0.71,31,"March review signals were present, but April i..."
3,False negative,19074,0.058,1,2063,0.291,19.30,22,The April threshold decline was not separated ...
4,False negative,19061,0.070,1,550,0.364,16.73,22,The April threshold decline was not separated ...
5,False negative,19060,0.070,1,655,0.458,12.72,22,The April threshold decline was not separated ...


These are errors against an observational month-to-month proxy, not confirmed editorial mistakes.


## 5. Self-check

- [ ] Warehouse contract and ML-07 candidate rule are unchanged.
- [ ] March features only; April decline proxy is a target, never an input.
- [ ] Client groups are disjoint across train and test; IDs are grouping-only.
- [ ] Rule and model are compared on the same held-out candidates with the same P@K, ROC-AUC, and average-precision metrics.
- [ ] Coefficients and anonymous model errors are interpreted cautiously; one split is not a performance guarantee.
- [ ] Results do not show page/client IDs, raw queries, or credentials.
- [ ] Notebook executed top to bottom under Python 3.11.

In [6]:
base_p50 = comparison.loc['ML-07 rule: log impressions', 'Precision@50']
model_p50 = comparison.loc['Logistic Regression', 'Precision@50']
base_auc = comparison.loc['ML-07 rule: log impressions', 'ROC-AUC']
model_auc = comparison.loc['Logistic Regression', 'ROC-AUC']
base_ap = comparison.loc['ML-07 rule: log impressions', 'Average precision']
model_ap = comparison.loc['Logistic Regression', 'Average precision']
print(f"On this client-held-out split, Precision@50 was {model_p50:.3f} for Logistic Regression versus {base_p50:.3f} for the ML-07 rule; ROC-AUC was {model_auc:.3f} versus {base_auc:.3f}, and average precision was {model_ap:.3f} versus {base_ap:.3f}.")
if model_p50 > base_p50 + 0.01 and model_auc > base_auc + 0.01 and model_ap > base_ap + 0.01:
    print('The model improved top-50 precision and both whole-pool ranking metrics on this split; investigate further, but do not treat one split as reliable future performance.')
elif model_p50 > base_p50 + 0.01:
    print('The model had higher top-50 precision on this split, but ROC-AUC and average precision did not materially improve. Treat this as a local top-of-queue result, not an overall ranking win; test more client folds before replacing the rule.')
else:
    print('The model did not improve top-50 precision on this split; the simpler ML-07 rule remains the stronger choice for this comparison.')
print('This analysis ranks only the ML-07 rule-eligible pool. The April proxy is retrospective, and the results do not show that refreshing content causes recovery.')
assert train_clients.isdisjoint(test_clients)
assert not set(ID_COLUMNS + [LABEL, 'impressions_apr']) & set(FEATURES)
assert len(comparison) == 2
print('Self-check passed: client separation, feature boundary, two-way comparison, and fixed-seed reproducibility.')

On this client-held-out split, Precision@50 was 0.720 for Logistic Regression versus 0.320 for the ML-07 rule; ROC-AUC was 0.496 versus 0.496, and average precision was 0.550 versus 0.550.
The model had higher top-50 precision on this split, but ROC-AUC and average precision did not materially improve. Treat this as a local top-of-queue result, not an overall ranking win; test more client folds before replacing the rule.
This analysis ranks only the ML-07 rule-eligible pool. The April proxy is retrospective, and the results do not show that refreshing content causes recovery.
Self-check passed: client separation, feature boundary, two-way comparison, and fixed-seed reproducibility.
